# Demo: Spend vs Exposure Variables in Frequentist MMM

**Duration:** ~30 min | **Level:** Intermediate | **Framework:** OLS (statsmodels)

---

### The core question

Our current OLS pipeline applies **adstock** and **saturation** directly to *spend* columns.  
But adstock models *memory decay after seeing an ad*, and saturation models *diminishing returns at high frequency* — both are properties of **exposure** (impressions), not dollars.

**Why does it matter?**  
CPMs fluctuate over time (Q4 inflation, auction dynamics, inventory scarcity). When CPMs rise, $1 buys fewer impressions. A spend-based model can't distinguish "we spent more" from "we reached more people."

### The causal chain

```
Spend  →  [Cost Curve]  →  Impressions  →  [Adstock]  →  [Saturation]  →  Revenue
```

This notebook demonstrates:

1. **CPM variability** — how the spend-to-impression ratio changes over time
2. **Model comparison** — spend-based vs. impression-based OLS
3. **The missing layer** — converting impression-based response curves back to budget allocation

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from utils import adstock_geometric, saturation_hill, build_ols_model, model_diagnostics

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Load the workshop data (real-world-style monthly data)
df = pd.read_csv("MMM_Workshop_Data.csv")
df["Month"] = pd.to_datetime(df["Month"], format="%b-%y")
print(f"Shape: {df.shape[0]} months × {df.shape[1]} columns")
df.head(3)

## Part 1 — CPM variability: why spend ≠ exposure

We pick three digital channels that have **both** spend and impression columns in the data:

| Channel | Spend column | Exposure column |
|---|---|---|
| Paid Search | `Paid_Search_Spends` | `Paid_Search_Impressions` |
| Programmatic Display | `Programmatic_Display_Spends` | `Programmatic_Display_Impressions` |
| Meta (aggregate) | `Meta_Spends_Agg` | `Meta_Agg_Impressions` |

In [ ]:
# Define our channel pairs
CHANNELS = {
    "Paid Search": {
        "spend": "Paid_Search_Spends",
        "impressions": "Paid_Search_Impressions",
    },
    "Prog. Display": {
        "spend": "Programmatic_Display_Spends",
        "impressions": "Programmatic_Display_Impressions",
    },
    "Meta (Agg)": {
        "spend": "Meta_Spends_Agg",
        "impressions": "Meta_Agg_Impressions",
    },
}

# Compute observed CPM for each channel (where both spend & impressions > 0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, (label, cols) in zip(axes, CHANNELS.items()):
    spend = df[cols["spend"]]
    imps = df[cols["impressions"]]
    mask = (spend > 0) & (imps > 0)
    cpm = spend[mask] / (imps[mask] / 1000)
    
    ax.plot(df["Month"][mask], cpm, "o-", markersize=4, linewidth=1.2)
    ax.set_title(f"{label}\nCPM  (CV = {cpm.std()/cpm.mean():.0%})", fontsize=11)
    ax.set_ylabel("$ per 1,000 impressions")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Observed CPM over time — cost of impressions is NOT constant",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

print("Key insight: Paid Search CPM varies by 55% (CV). Using spend as a proxy")
print("for exposure in the adstock/saturation pipeline ignores this variation.")

## Part 2 — Head-to-head: Spend-based vs Impression-based OLS

We build two simple OLS models predicting `Sales_Revenue_Total`:

- **Model A (Spend):** Adstock + Hill saturation applied to *spend* columns
- **Model B (Exposure):** Same transforms applied to *impression* columns

Both use the same controls (inflation, competitor ATL spends) and the same adstock/saturation hyperparameters, so the *only* difference is the unit of the media input.

In [ ]:
# --- Shared hyperparameters (kept simple for demo) ---
THETA = {"Paid Search": 0.2, "Prog. Display": 0.1, "Meta (Agg)": 0.15}
ALPHA = 1.5   # Hill shape
GAMMA = 0.5   # Hill inflection point

# Dependent variable
y = df["Sales_Revenue_Total"].values

# Control variables (same for both models)
controls = df[["Inflation_Rate", "Brand_B_ATL_Spends", "Brand_PH_ATL_Spends"]].copy()

def transform_channel(series, theta):
    """Adstock → Hill saturation pipeline for a single channel."""
    ad = adstock_geometric(series, theta)["x_decayed"]
    return saturation_hill(ad, ALPHA, GAMMA)

# --- Model A: Spend-based ---
X_spend = controls.copy()
for label, cols in CHANNELS.items():
    raw = df[cols["spend"]].values
    X_spend[f"{label}_sat"] = transform_channel(raw, THETA[label])

# --- Model B: Impression-based ---
X_imps = controls.copy()
for label, cols in CHANNELS.items():
    raw = df[cols["impressions"]].values
    X_imps[f"{label}_sat"] = transform_channel(raw, THETA[label])

model_spend = build_ols_model(y, X_spend)
model_imps  = build_ols_model(y, X_imps)

# Compare
comparison = pd.DataFrame({
    "Metric": ["R²", "Adj R²", "AIC", "MAPE", "NRMSE", "Durbin-Watson"],
    "Model A (Spend)": [
        f"{model_spend.rsquared:.4f}",
        f"{model_spend.rsquared_adj:.4f}",
        f"{model_spend.aic:.1f}",
        f"{np.mean(np.abs((y - model_spend.fittedvalues)/y)):.2%}",
        f"{np.sqrt(np.mean((y - model_spend.fittedvalues)**2))/(y.max()-y.min()):.4f}",
        f"{sm.stats.stattools.durbin_watson(model_spend.resid):.2f}",
    ],
    "Model B (Impressions)": [
        f"{model_imps.rsquared:.4f}",
        f"{model_imps.rsquared_adj:.4f}",
        f"{model_imps.aic:.1f}",
        f"{np.mean(np.abs((y - model_imps.fittedvalues)/y)):.2%}",
        f"{np.sqrt(np.mean((y - model_imps.fittedvalues)**2))/(y.max()-y.min()):.4f}",
        f"{sm.stats.stattools.durbin_watson(model_imps.resid):.2f}",
    ],
})
print("Model Comparison: Spend vs Impression inputs")
print("=" * 55)
comparison

In [ ]:
# Visual comparison: fitted vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (model, title) in zip(axes, [
    (model_spend, "Model A — Spend-based"),
    (model_imps,  "Model B — Impression-based"),
]):
    ax.plot(df["Month"], y, "k-", label="Actual", linewidth=1.5)
    ax.plot(df["Month"], model.fittedvalues, "--", label="Fitted", linewidth=1.5)
    ax.set_title(f"{title}  (R² = {model.rsquared:.4f})", fontsize=11)
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Sales Revenue")

fig.suptitle("Actual vs Fitted: which input captures the signal better?",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Side-by-side coefficient comparison
coefs_spend = model_spend.params.drop("const")
coefs_imps  = model_imps.params.drop("const")
pvals_spend = model_spend.pvalues.drop("const")
pvals_imps  = model_imps.pvalues.drop("const")

coef_df = pd.DataFrame({
    "Variable": coefs_spend.index,
    "Coef (Spend)": [f"{v:.2f}" for v in coefs_spend.values],
    "p-val (Spend)": [f"{v:.4f}" for v in pvals_spend.values],
    "Coef (Imps)": [f"{v:.2f}" for v in coefs_imps.values],
    "p-val (Imps)": [f"{v:.4f}" for v in pvals_imps.values],
})
print("Coefficient comparison")
print("=" * 75)
coef_df

## Part 3 — The missing layer: from impressions back to budget

The impression-based model tells us: *"one more saturated-impression-unit of Paid Search is worth β dollars of revenue."*

But a CMO asks: *"should I add $1,000 to Paid Search or Meta next month?"*

To answer that, we need a **cost curve** — the relationship between spend and impressions.  
In its simplest form, this is just the observed CPM:

$$\text{Impressions}_t = \frac{\text{Spend}_t}{\text{CPM}_t / 1000}$$

For budget **optimization**, we invert the full chain:

$$\text{Marginal Revenue per } \$ = \frac{\partial \text{Revenue}}{\partial \text{Impressions}} \times \frac{\partial \text{Impressions}}{\partial \text{Spend}}$$

The first term comes from our impression-based model. The second term comes from the cost curve (1/CPM × 1000).

This is the key insight: the **response model** and the **cost model** are separate concerns.  When CPMs are stable, conflating them (i.e., using spend directly) is a reasonable approximation.  When CPMs vary, it's not.

In [ ]:
# --- Marginal ROI comparison: spend-based vs impression-based ---
# For the impression-based model, marginal ROI depends on the current CPM.
# We compute it at the *most recent* CPM and at the *average* CPM.

results = []
for label, cols in CHANNELS.items():
    var = f"{label}_sat"
    
    # Model A (spend): coefficient IS the marginal response per $ (after transform)
    beta_spend = model_spend.params[var]
    
    # Model B (impressions): coefficient is marginal response per impression-unit
    beta_imps = model_imps.params[var]
    
    # Compute CPM stats
    spend = df[cols["spend"]]
    imps  = df[cols["impressions"]]
    mask  = (spend > 0) & (imps > 0)
    cpm   = spend[mask] / (imps[mask] / 1000)
    
    cpm_recent = cpm.iloc[-1] if len(cpm) > 0 else cpm.mean()
    cpm_avg    = cpm.mean()
    
    # Marginal revenue per $1 of spend (via impression model + cost curve)
    # d(Revenue)/d(Spend) = beta_imps * d(Impressions)/d(Spend)
    #                     = beta_imps * (1000 / CPM)
    mroi_via_recent_cpm = beta_imps * (1000 / cpm_recent)
    mroi_via_avg_cpm    = beta_imps * (1000 / cpm_avg)
    
    results.append({
        "Channel": label,
        "β (Spend model)": f"{beta_spend:.1f}",
        "β (Imps model)": f"{beta_imps:.4f}",
        "Avg CPM ($)": f"{cpm_avg:.2f}",
        "Recent CPM ($)": f"{cpm_recent:.2f}",
        "mROI (Spend model)": f"{beta_spend:.1f}",
        "mROI (Imps @ avg CPM)": f"{mroi_via_avg_cpm:.1f}",
        "mROI (Imps @ recent CPM)": f"{mroi_via_recent_cpm:.1f}",
    })

roi_df = pd.DataFrame(results)
print("Marginal ROI: spend-based model vs impression-based model + cost curve")
print("=" * 80)
roi_df

In [ ]:
# Visualize: how mROI shifts with CPM for the impression-based model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (label, cols) in zip(axes, CHANNELS.items()):
    var = f"{label}_sat"
    beta_imps = model_imps.params[var]
    beta_spend = model_spend.params[var]
    
    spend = df[cols["spend"]]
    imps  = df[cols["impressions"]]
    mask  = (spend > 0) & (imps > 0)
    cpm   = spend[mask] / (imps[mask] / 1000)
    
    # mROI over time using impression model + observed CPM
    mroi_time = beta_imps * (1000 / cpm)
    
    ax.plot(df["Month"][mask], mroi_time, "o-", markersize=4, linewidth=1.2,
            color="steelblue", label="Imps model (CPM-adjusted)")
    ax.axhline(beta_spend, color="firebrick", linestyle="--", linewidth=1.5,
               label=f"Spend model (fixed: {beta_spend:.0f})")
    ax.set_title(label, fontsize=11)
    ax.set_ylabel("mROI ($ revenue / $ spend)")
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("mROI over time: the impression model captures CPM-driven variation",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

print("The spend model gives a single, time-invariant mROI per channel.")
print("The impression model's mROI moves with CPM — allocate more when impressions are cheap.")

## Part 4 — Bonus: same story on synthetic data (ground truth available)

The synthetic data generator (`generate_synthetic_data.py`) now creates the data with the **correct causal structure**: revenue is driven by impressions (after adstock + saturation), not spend. This means we *know* the impression-based model should recover the true DGP better.

Let's verify with one geo.

In [ ]:
# Load synthetic data — filter to one geo for simplicity
syn = pd.read_csv("synthetic_multi_geo_data.csv")
geo_df = syn[syn["geo"] == "west"].reset_index(drop=True)

y_syn = geo_df["revenue"].values

# Channels in synthetic data
syn_channels = ["tv", "digital", "social", "search", "display"]
# True theta values used in the DGP
true_theta = {"tv": 0.70, "digital": 0.10, "social": 0.30, "search": 0.15, "display": 0.25}

controls_syn = geo_df[["price_index", "competitor_spend"]].copy()

# Model A: spend-based
X_syn_spend = controls_syn.copy()
for ch in syn_channels:
    X_syn_spend[ch] = transform_channel(geo_df[f"{ch}_spend"].values, true_theta[ch])

# Model B: impression-based
X_syn_imps = controls_syn.copy()
for ch in syn_channels:
    X_syn_imps[ch] = transform_channel(geo_df[f"{ch}_impressions"].values, true_theta[ch])

m_syn_spend = build_ols_model(y_syn, X_syn_spend)
m_syn_imps  = build_ols_model(y_syn, X_syn_imps)

print("Synthetic data (geo=west) — the DGP uses impressions, so Model B should win:")
print(f"  Model A (Spend):       R² = {m_syn_spend.rsquared:.4f}   Adj R² = {m_syn_spend.rsquared_adj:.4f}")
print(f"  Model B (Impressions): R² = {m_syn_imps.rsquared:.4f}   Adj R² = {m_syn_imps.rsquared_adj:.4f}")
print(f"\n  ΔR² = {m_syn_imps.rsquared - m_syn_spend.rsquared:+.4f}  (positive means impressions model is better)")

In [ ]:
# Show CPM variability in synthetic data too
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))

dates_syn = pd.to_datetime(geo_df["date"])
for ax, ch in zip(axes, syn_channels):
    cpm = geo_df[f"{ch}_cpm"]
    ax.plot(dates_syn, cpm, linewidth=1.2)
    ax.set_title(f"{ch}\nCV = {cpm.std()/cpm.mean():.0%}", fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.set_ylabel("CPM ($)")

fig.suptitle("Synthetic data: time-varying CPMs (geo=west)", fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## Takeaways

| | Spend-based model | Exposure-based model |
|---|---|---|
| **What adstock/saturation model** | Dollar carry-over & saturation | Impression carry-over & saturation |
| **Theoretically correct?** | Proxy — works when CPMs are stable | Yes — models the actual causal mechanism |
| **Budget allocation** | Direct (coeff = $/$ ROI) | Requires cost curve layer (CPM) |
| **When to prefer** | Quick-and-dirty; no impression data | CPMs vary; you need time-varying ROI |
| **Who does this** | Many lightweight MMMs | Meridian, Robyn (under the hood), academic MMMs |

**Bottom line:** If you have impression data and CPMs vary meaningfully, model the *exposure* and add a cost-curve layer for budget allocation. If you only have spend data or CPMs are stable, the spend-based shortcut is a reasonable approximation — just know what you're approximating.

### Next steps for the course

- Session 3 (OLS MMM) can be extended to let students choose spend vs. impression inputs and compare
- The `config_file.csv` could benefit from a `variable_type` column to distinguish spend from exposure
- Meridian (Week 3) already handles this split natively via `media_spend` vs. `media` arrays